# FULL-SYN — chạy thử M5 trên task 2

[Phân tích round 1](../../docs/project/full-syn-r1-analysis-2026-10-29.md), mục 3.6 và 7. Giả thuyết: model nhầm các SKU
anh em vì backbone và encoder bị khoá sau FSA, mà FSA chỉ học SKU của task 1. M5 cho tầng cuối của backbone
(`stages_3`) và encoder học tiếp từ task 2 ở `lr_old`, kèm một loss giữ output của encoder gần với model thầy
(`--feature_kd`). Mọi thứ khác giữ như round 1, kể cả learning rate và số epoch, để chỉ đổi một biến.

Notebook này là **log của lần chạy thử**: FSA và PDP task 1 lấy nguyên từ round 1 (`seed_campaign`), chỉ train
**task 2**, rồi so với task 2 của round 1 trên val:

| Kết quả | Quyết định |
|---|---|
| cAcc-M2 tăng ≥ 3 điểm **và** AP50 các SKU task 1 giảm ≤ 1 điểm | `go`: làm lại prototype của SKU task 1, rồi chạy task 3–5 |
| cAcc-M2 tăng < 3 điểm | `stop`: giả thuyết sai, chuyển sang train ở 800px (hướng B) |
| Tăng nhưng SKU task 1 giảm > 1 điểm | `retry`: thử lại một lần với `feature_kd` lớn hơn |

Code chặn chạy quá task 2 và chặn chấm test, vì prototype của SKU task 1 phải được làm lại trước.
Bị ngắt giữa chừng thì chạy lại chính file này: bước đã xong được bỏ qua.

In [ ]:
import json, os, sys
from pathlib import Path
REPO = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'pyproject.toml').is_file())
sys.path.insert(0, str(REPO))
from autocheckout import full_syn
from autocheckout.io import load_json
from autocheckout.training_workflow import resolve
SETTINGS_PATH = resolve(os.environ.get('AUTOCHECKOUT_TRAINING_SETTINGS', 'data/training/settings_full_syn_m5_pilot.json'))
SETTINGS = load_json(SETTINGS_PATH)
PLAN = full_syn.prepare(SETTINGS)
print('Settings:', SETTINGS_PATH)
print('Campaign:', PLAN['root'], '| round', PLAN['round'], '| dry run' if PLAN['dry_run'] else '| real run')
print('Synthetic config sha256:', PLAN['release_config_sha256'])
print('Train images per task:', PLAN['train_images'])
print('PDP:', PLAN['pdp_experiment'], '| FSA và task 1 lấy từ:', PLAN['seed_root'], PLAN['seed'])
print(json.dumps({k: PLAN[k] for k in ('resolution', 'batch_size', 'effective_batch', 'fsa_batch_size', 'num_workers', 'tf32',
                                         'fsa_epochs', 'epochs', 'audit_images')}, indent=1))

In [ ]:
import platform, subprocess, torch
from autocheckout.runinfo import git_state
git = git_state(REPO)
print('git', git.get('commit'), '| dirty' if git.get('dirty') else '| clean')
print('Python', platform.python_version(), '| torch', torch.__version__, '| CUDA', torch.version.cuda)
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none')
print(subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total,driver_version', '--format=csv'],
                     capture_output=True, text=True).stdout if torch.cuda.is_available() else '')

## FSA và task 1 từ round 1

Link `runs/EXP-FULL-SYN-FSA` và `task_1`, chép báo cáo `fsa`/`task1` và policy đếm của stage 1. Kiểm md5 của
`task_1/task_final.pth` so với lúc chuẩn bị campaign.

In [ ]:
SEED = full_syn.stage_seed(PLAN)
print(json.dumps({stage: {k: SEED[stage][k] for k in ('cAcc_m2', 'AP50', 'gate')} for stage in SEED}, indent=1))

## Task 2 với M5

Trước khi train: audit nhãn giả của model task 1 (như round 1, cổng precision ≥ 0,5). Sau khi train: chấm val.

In [ ]:
TASKS = full_syn.stage_tasks(PLAN, last=2)
NEW = TASKS['tasks'][2]
print(f"cAcc M2 {NEW['cAcc_m2']:.4f} | M1 {NEW['cAcc_m1_groups']:.4f} (nhóm) | AP50 {NEW['AP50']} | {NEW['train_hours']} h")
print('Nhầm lẫn nhiều nhất:', NEW['confusions_top20'][:10])

## So với task 2 của round 1

In [ ]:
R1 = load_json(Path(PLAN['seed_root']) / 'stages/tasks.json')['tasks']['2']
decision, reason = full_syn.pilot_verdict(NEW, R1)
print(decision.upper(), '-', reason)
r1_pairs = {(c['true'], c['predicted']): c['count'] for c in R1['confusions_top20']}
for c in NEW['confusions_top20'][:10]:
    print(f"  {c['true']:>3} -> {c['predicted']:>3}: {c['count']:>4} (round 1: {r1_pairs.get((c['true'], c['predicted']), '-')})")